# Trabalho Final de Fundamentos de Banco de Dados

## Equipe

- Anna Liz Veríssimo Mendes
- Gabriel Fernandes de Sousa
- Kauan Ferreira da Silva

# Bibliotecas usadas no projeto

In [1]:
import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy as sq
from sqlalchemy import create_engine
import panel as pn

# Conexão com banco de dados

## Carregar dados do .env

In [2]:
load_dotenv()

True

In [3]:
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')

## Conectar-se com banco de dados

In [4]:
con = pg.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)

In [ ]:
cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'

create_engine(cnx)

# Atributos da Tabela: Tarefas

In [6]:
id_tarefa = pn.widgets.IntInput(
    name='ID da Tarefa',
    value=0,
    disabled=True # Isso retira o campo de input, tornando-o apenas leitura
)
data_prazo = pn.widgets.DatePicker(
    name='Data',
    width=250,
    disabled=False
)

titulo = pn.widgets.TextInput(
    name='Título',
    placeholder='Digite o título da tarefa')

descricao = pn.widgets.TextAreaInput(
    name='Descrição',
    placeholder='Digite a descrição da tarefa')

id_evento = pn.widgets.IntInput(
    name='ID do Evento',
    value=0,
    disabled=True
)

id_colaborador = pn.widgets.IntInput(
    name='ID do Colaborador',
    value=0,
    disabled=True
)


# Criando Interface

In [ ]:
pn.extension()
pn.extension('tabulator')
pn.extension(notifications=True)

# Telas

### Botões

In [8]:
buttonAtualizar = pn.widgets.Button(name='Atualizar', button_type='warning')
# cria o botão de atualizar com o tipo de botão "warning" (aviso)

### Funções de query

In [9]:
def consultar_tarefas():
    # Consulta todas as tarefas no banco de dados e retorna um DataFrame
    query = "SELECT * FROM tarefa ORDER BY id_tarefa"
    df = pd.read_sql(query, con)
    return df

def atualizar_tarefa(titulo, descricao, data_prazo, id_tarefa):
    query = """
        UPDATE tarefa
        SET titulo = %s, descricao = %s, data_prazo = %s WHERE id_tarefa = %s
    """
    values = (titulo, descricao, data_prazo, id_tarefa)
    try:
        with con.cursor() as cursor:
            cursor.execute(query, values)
        con.commit()
    except Exception:
        con.rollback()
        raise


### Chamada de funções de query

In [10]:
def consultar(event=None):
    # Consulta todas as tarefas e atualiza a tabela
    tabela.value = consultar_tarefas()

def atualizar(event=None):
    try:
        atualizar_tarefa(
            titulo.value,
            descricao.value,
            data_prazo.value,
            id_tarefa.value
        )
        pn.state.notifications.success('Tarefa atualizada com sucesso!')
        consultar()
    except Exception as e:
        pn.state.notifications.error(f'Erro ao atualizar tarefa: {e}')

def selecionar_linha(event):
    # Atualiza os campos de input com os valores da tupla selecionada na tabela
    if event.row is None:
        return
    linha = tabela.value.iloc[event.row]
    id_tarefa.value = int(linha['id_tarefa'])
    titulo.value = linha['titulo']
    descricao.value = linha['descricao'] or ''
    data_prazo.value = linha['data_prazo']
    id_evento.value = int(linha['id_evento'])
    id_colaborador.value = int(linha['id_colaborador'])

In [ ]:
buttonAtualizar.on_click(atualizar)
# Associa a função de atualização ao botão(ao clicar)

In [ ]:
tabela = pn.widgets.Tabulator(consultar_tarefas(), disabled=True, show_index=False)
tabela.on_click(selecionar_linha)

In [13]:
form = pn.Column(
    id_tarefa,
    titulo,
    descricao,
    data_prazo,
    id_evento,
    id_colaborador,
    pn.Row(buttonAtualizar),
    width=300
)

layout = pn.Row(
    form,
    pn.Column(tabela, sizing_mode="stretch_width"),
    sizing_mode="stretch_width"
)

In [ ]:
layout.show()